# API_Test_AM — Neon API + Database smoke test + Phase 1 descriptives

**Author:** Aidan Meyers · Melaram Lab, TAMU-CC 
**Date created:** 2026-04-22 
**Pipeline version:** 0.3.5

This notebook tests connectivity to the South Texas Air Quality Neon database, validates that all 7 tables in the `aq` schema are present and queryable, runs Phase 1 Week 1 descriptive statistics, and produces three starter figures.

**Estimated runtime:** ~2 minutes

**Prerequisites:**
1. In Colab, click the 🔑 (key) icon in the left sidebar
2. **Add new secret** — Name: `AQ_POSTGRES_URL`, Value: paste your Neon connection string from https://console.neon.tech/app/projects/aged-salad-62359207
3. Toggle **Notebook access** ON for this notebook

**Documentation:** https://aidanjmeyers.github.io/south-texas-aq-pipeline/17_colab_database_guide/

## Setup — install dependencies

In [ ]:
!pip install -q "psycopg[binary]" sqlalchemy pandas matplotlib seaborn requests

## Cell 2 — Connect to Neon (direct SQL via SQLAlchemy)

If this prints `Connected. Server: PostgreSQL 16.x ...` you are good. Neon auto-pauses after 5 min idle, so the very first query may take ~500 ms to wake the compute — `pool_pre_ping=True` handles this transparently.

In [ ]:
from google.colab import userdata
from sqlalchemy import create_engine
import pandas as pd

engine = create_engine(
    userdata.get('AQ_POSTGRES_URL'),
    pool_pre_ping=True,
)
print('Connected. Server:', pd.read_sql('SELECT version()', engine).iloc[0, 0])

## Test 1 — Confirm every table in the `aq` schema is present

Expected output (one row per table, sorted by size descending):

| table_name | size | rows |
|---|---|---:|
| pollutant_hourly | 1408 MB | 7,699,105 |
| weather_hourly | 631 MB | 1,470,049 |
| aq_weather_daily | 132 MB | 390,738 |
| pollutant_daily | 69 MB | 390,738 |
| pollutant_monthly | 1880 kB | 11,333 |
| naaqs_design_values | 200 kB | 764 |
| site_registry | 64 kB | 47 |

In [ ]:
sql = """
SELECT
    table_name,
    pg_size_pretty(pg_total_relation_size(('aq.' || table_name)::regclass)) AS size,
    (xpath('/row/c/text()',
           query_to_xml(format('SELECT count(*) AS c FROM aq.%I', table_name),
                        false, true, '')))[1]::text::bigint AS rows
FROM information_schema.tables
WHERE table_schema = 'aq'
ORDER BY pg_total_relation_size(('aq.' || table_name)::regclass) DESC
"""
overview = pd.read_sql(sql, engine)
print(overview.to_string(index=False))

## Test 2 — Site registry sanity check

Should be exactly 47 sites broken down as: **42 active**, **3 reference** (CPS Energy fence-line), **1 disabled** (Williams Park), **1 excluded** (Calaveras Lake Park — TSP only, officially retired).

In [ ]:
sites = pd.read_sql("""
    SELECT data_status, COUNT(*) AS n_sites
    FROM aq.site_registry
    GROUP BY data_status
    ORDER BY n_sites DESC
""", engine)
print(sites)

## Test 3 — Per-pollutant non-null rates

Confirms the hourly tables have the right pollutant coverage. As of v0.3.5 the non-null rates are: VOCs 100%, PM10 98%, O3 97%, PM2.5 95%, NOx 95%, CO 91%, SO2 88%.

In [ ]:
non_null = pd.read_sql("""
    SELECT pollutant_group,
           COUNT(*)                              AS rows,
           COUNT(DISTINCT aqsid)                 AS sites,
           ROUND(
             (SUM(CASE WHEN sample_measurement IS NOT NULL THEN 1 ELSE 0 END)::numeric
              / COUNT(*) * 100), 2
           ) AS pct_non_null
    FROM aq.pollutant_hourly
    GROUP BY pollutant_group
    ORDER BY rows DESC
""", engine)
print(non_null.to_string(index=False))

## Phase 1 Week 1 — Descriptive statistics by pollutant × county × year

This is the manuscript-ready descriptives table. Saves to `descriptives_pollutant_county_year.csv`.

In [ ]:
sql = """
SELECT
    pollutant_group,
    county_name,
    EXTRACT(YEAR FROM date_local::date)::int AS year,
    COUNT(*)                                                                            AS n_obs,
    ROUND(AVG(sample_measurement)::numeric, 4)                                          AS mean,
    ROUND(STDDEV(sample_measurement)::numeric, 4)                                       AS sd,
    ROUND(PERCENTILE_CONT(0.05) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS p5,
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS p25,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS median,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS p75,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY sample_measurement)::numeric, 4) AS p95,
    ROUND(MAX(sample_measurement)::numeric, 4)                                          AS max
FROM aq.pollutant_hourly
WHERE sample_measurement IS NOT NULL
GROUP BY pollutant_group, county_name, year
ORDER BY pollutant_group, county_name, year
"""
descriptives = pd.read_sql(sql, engine)
descriptives.to_csv('descriptives_pollutant_county_year.csv', index=False)
print(f'{len(descriptives):,} (pollutant x county x year) rows')
descriptives.head(15)

## Figure 1 — Annual mean by pollutant and county

One panel per pollutant group, one line per county. Saved to `fig1_annual_means_by_pollutant_county.png`.

In [ ]:
import matplotlib.pyplot as plt

pollutants = sorted(descriptives.pollutant_group.unique())
fig, axes = plt.subplots(len(pollutants), 1,
                          figsize=(10, 3 * len(pollutants)),
                          sharex=True)
if len(pollutants) == 1:
    axes = [axes]

for ax, p in zip(axes, pollutants):
    sub = descriptives[descriptives.pollutant_group == p]
    for county, grp in sub.groupby('county_name'):
        ax.plot(grp.year, grp['mean'], marker='o', linewidth=1.5, label=county)
    ax.set_title(f'{p} - annual mean by county', loc='left', color='#213c4e')
    ax.set_ylabel('mean concentration')
    ax.grid(alpha=0.3)

axes[-1].set_xlabel('Year')
axes[0].legend(loc='upper left', bbox_to_anchor=(1.01, 1), fontsize=8)
plt.tight_layout()
plt.savefig('fig1_annual_means_by_pollutant_county.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2 — Diurnal ozone profile at Camp Bullis (sanity check)

Should show the classic photochemistry curve: ozone climbs after sunrise, peaks 2–4 PM, declines through the evening. The dashed red line is the 8-hr NAAQS (0.070 ppm).

In [ ]:
diurnal = pd.read_sql("""
    SELECT EXTRACT(HOUR FROM datetime)::int AS hour_of_day,
           AVG(sample_measurement) AS mean_o3_ppm,
           PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY sample_measurement) AS p95_o3
    FROM aq.pollutant_hourly
    WHERE aqsid = '480290052'
      AND pollutant_group = 'Ozone'
      AND year IN (2022, 2023, 2024)
      AND sample_measurement IS NOT NULL
    GROUP BY hour_of_day
    ORDER BY hour_of_day
""", engine)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(diurnal.hour_of_day, diurnal.mean_o3_ppm,
        marker='o', linewidth=2, color='#213c4e', label='Mean')
ax.plot(diurnal.hour_of_day, diurnal.p95_o3,
        marker='^', linewidth=2, color='#c2410c', label='95th percentile')
ax.axhline(0.070, color='red', linestyle='--', alpha=0.6,
           label='8-hr NAAQS 0.070 ppm')
ax.set_xlabel('Hour of day (local)')
ax.set_ylabel('Ozone (ppm)')
ax.set_title('Camp Bullis diurnal ozone profile, 2022-2024 average', color='#213c4e')
ax.set_xticks(range(0, 24, 3))
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig('fig2_diurnal_ozone_camp_bullis.png', dpi=150)
plt.show()

## Figure 3 — Hourly PM₂.₅ distribution by county (BREATHE-CC relevant)

Box plots ordered by median PM₂.₅. Red dashed line is the revised annual NAAQS (9.0 µg/m³, Feb 2024).

In [ ]:
import seaborn as sns

pm25 = pd.read_sql("""
    SELECT county_name, sample_measurement AS pm25
    FROM aq.pollutant_hourly
    WHERE pollutant_group = 'PM2.5'
      AND year IN (2022, 2023, 2024)
      AND sample_measurement BETWEEN 0 AND 200
""", engine)

county_order = (pm25.groupby('county_name')['pm25']
                .median().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=pm25, x='county_name', y='pm25', order=county_order,
            color='#345372', fliersize=1, ax=ax)
ax.axhline(9.0, color='red', linestyle='--', alpha=0.7,
           label='Annual NAAQS 9.0 ug/m^3')
ax.set_xlabel('County')
ax.set_ylabel('PM2.5 (ug/m^3)')
ax.set_title('Hourly PM2.5 distribution by county, 2022-2024', color='#213c4e')
ax.set_ylim(0, 50)
plt.xticks(rotation=30, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('fig3_pm25_boxplot_by_county.png', dpi=150)
plt.show()

## Bonus — Data API smoke test (HTTP path, no Postgres driver)

Tests the **anonymous** PostgREST role over plain HTTP. If this returns rows, the public API is live and the SELECT grant on `aq.naaqs_design_values` is in place. If you set up an `AQ_NEON_JWT` Colab secret, you can also test the **authenticated** path by uncommenting the auth header line below.

**Auth login URL** (to mint a JWT): https://ep-muddy-star-ant3mvxo.neonauth.c-6.us-east-1.aws.neon.tech/neondb/auth

In [ ]:
import requests

API = 'https://ep-muddy-star-ant3mvxo.apirest.c-6.us-east-1.aws.neon.tech/neondb/rest/v1'

headers = {'Accept': 'application/json'}
# Uncomment for authenticated mode (requires AQ_NEON_JWT Colab secret):
# headers['Authorization'] = f"Bearer {userdata.get('AQ_NEON_JWT')}"

resp = requests.get(
    f'{API}/aq.naaqs_design_values',
    params={
        'year': 'eq.2023',
        'exceeds': 'is.true',
        'select': 'aqsid,site_name,county_name,metric,value',
        'order': 'value.desc',
        'limit': '10',
    },
    headers=headers,
)
print('HTTP status:', resp.status_code)
if resp.ok:
    df_api = pd.DataFrame(resp.json())
    print(df_api)
else:
    print('Body:', resp.text[:500])

## Summary

| Cell | Validates |
|---|---|
| Cell 2 | Direct SQL connection works; Neon waking from auto-pause |
| Test 1 | All 7 `aq.*` tables present + accessible |
| Test 2 | Site registry breakdown matches the 47-site inventory |
| Test 3 | Hourly tables have expected non-null rates per pollutant |
| Phase 1 descriptives | Per-pollutant × county × year stats compute cleanly |
| Figure 1 | County-grouped trends render |
| Figure 2 | Hourly ozone shows expected photochemistry |
| Figure 3 | PM₂.₅ exceedance rate visible vs. revised NAAQS |
| Bonus | Data API + anonymous role + PostgREST query syntax all work |

**Files written to the Colab runtime (download via the file panel):**

- `descriptives_pollutant_county_year.csv`
- `fig1_annual_means_by_pollutant_county.png`
- `fig2_diurnal_ozone_camp_bullis.png`
- `fig3_pm25_boxplot_by_county.png`

**Next steps:** see [doc 16 — analysis timeline](https://aidanjmeyers.github.io/south-texas-aq-pipeline/16_project_timeline/) for the full Week 1 task list.